In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
%sql
use catalog projectv1

In [0]:
%sql
use schema projectschema

### **Data Reading From Source**

In [0]:
df = spark.sql("select * from tblProductsSilver")

In [0]:
%sql
describe tblProductsSilver


#  **Dividing New Records vs Old Records**

In [0]:
%sql
CREATE table DimProducts
(
  DimProductKey BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id string,
  ProductName string,
  ProductCategory string,
  price double,
  DiscountPrice double,
  create_date date,
  update_date date
)

In [0]:
from delta.tables import DeltaTable

In [0]:
%sql
Merge INTO DimProducts
Using tblProductsSilver
ON DimProducts.product_id = tblProductsSilver.product_id
WHEN MATCHED THEN UPDATE SET DimProducts.price = tblProductsSilver.price, DimProducts.DiscountPrice = tblProductsSilver.discounted_price, DimProducts.update_date = current_timestamp()  -- changing attributes

WHEN NOT MATCHED THEN INSERT( product_id, ProductName, ProductCategory, price, DiscountPrice, create_date)
VALUES (tblProductsSilver.product_id, tblProductsSilver.ProductName, tblProductsSilver.ProductCategory, tblProductsSilver.price, tblProductsSilver.discounted_price, current_timestamp())
;
     

In [0]:
%sql
SELECT * FROM dimproducts